# Multinomial Naive Bayes с нуля: классификация спама

В этом ноутбуке мы:

1. вспомним формулу Байеса;
2. разберем термины «априорная вероятность», «правдоподобие» и «апостериорная вероятность»;
3. вручную обучим Multinomial Naive Bayes на небольшом наборе сообщений;
4. вручную классифицируем новое сообщение;
5. применим сглаживание Лапласа;
6. повторим вычисления в логарифмах;
7. проверим результат с помощью `scikit-learn`;
8. сравним Multinomial Naive Bayes с Bernoulli Naive Bayes.

Для краткости сообщения, не являющиеся спамом, будем обозначать как **Ham**.

## 1. Формула Байеса

Пусть:

- $C$ — класс сообщения, например `Spam` или `Ham`;
- $d$ — наблюдаемое сообщение.

Формула Байеса:

$$
P(C\mid d)=\frac{P(d\mid C)P(C)}{P(d)}.
$$

Здесь:

- $P(C)$ — **априорная вероятность** класса;
- $P(d\mid C)$ — **правдоподобие**, то есть вероятность увидеть сообщение $d$, если известен его класс;
- $P(d)$ — полная вероятность наблюдаемого сообщения;
- $P(C\mid d)$ — **апостериорная вероятность** класса после наблюдения сообщения.

Для нескольких классов знаменатель можно записать так:

$$
P(d)=\sum_{C'}P(d\mid C')P(C').
$$

Поэтому:

$$
P(C\mid d)=
\frac{P(d\mid C)P(C)}
{\sum_{C'}P(d\mid C')P(C')}.
$$

При классификации знаменатель одинаков для всех классов. Поэтому сначала можно сравнить ненормированные оценки:

$$
P(C\mid d)\propto P(C)P(d\mid C).
$$

Правило классификации, или MAP-оценка:

$$
\hat C=
\arg\max_C P(C\mid d)
=
\arg\max_C P(C)P(d\mid C).
$$

## 2. Multinomial Naive Bayes

Multinomial Naive Bayes используется, когда признаки являются счетчиками. В текстовой классификации признаком обычно является количество вхождений слова в документ.

Модель делает два основных предположения:

1. **мешок слов** — порядок слов не учитывается;
2. **условная независимость** — слова считаются независимыми при условии класса.

Пусть:

- $V$ — словарь;
- $n_w$ — количество вхождений слова $w$ в документ;
- $P(w\mid C)$ — вероятность слова $w$ в классе $C$.

Тогда правдоподобие документа записывается как:

$$
P(d\mid C)=\prod_{w\in V}P(w\mid C)^{n_w}.
$$

Полная мультиномиальная формула также содержит коэффициент:

$$
P(d\mid C)=
\frac{n!}{\prod_{w\in V}n_w!}
\prod_{w\in V}P(w\mid C)^{n_w},
$$

где $n$ — общее количество слов в документе.

При сравнении классов мультиномиальный коэффициент можно не учитывать, потому что он не зависит от класса.

Априорная вероятность класса оценивается как доля документов этого класса:

$$
P(C)=\frac{N_C}{N},
$$

где $N_C$ — количество документов класса $C$, а $N$ — общее количество документов.

Вероятность слова без сглаживания:

$$
P(w\mid C)=
\frac{N(w,C)}
{\sum_{w'\in V}N(w',C)}.
$$

## 3. Обучающая выборка

Рассмотрим четыре сообщения:

| ID | Сообщение | Класс |
|---|---|---|
| M1 | `buy cheap now` | Spam |
| M2 | `limited offer buy` | Spam |
| M3 | `meet me now` | Ham |
| M4 | `catch up soon` | Ham |

Будем классифицировать новое сообщение:

> `buy now`

In [ ]:
import pandas as pd

data = pd.DataFrame({
    "id": ["M1", "M2", "M3", "M4"],
    "text": [
        "buy cheap now",
        "limited offer buy",
        "meet me now",
        "catch up soon"
    ],
    "label": ["spam", "spam", "ham", "ham"]
})

data

,id,text,label
0,M1,buy cheap now,spam
1,M2,limited offer buy,spam
2,M3,meet me now,ham
3,M4,catch up soon,ham


## 4. Построение словаря

Словарь состоит из всех уникальных слов обучающей выборки:

$$
V=
\{
\text{buy, cheap, now, limited, offer, meet, me, catch, up, soon}
\}.
$$

Размер словаря:

$$
|V|=10.
$$

In [ ]:
from collections import Counter

def tokenize(text):
    return text.lower().split()

vocabulary = sorted({
    word
    for text in data["text"]
    for word in tokenize(text)
})

print("Словарь:", vocabulary)
print("Размер словаря:", len(vocabulary))

Словарь: ['buy', 'catch', 'cheap', 'limited', 'me', 'meet', 'now', 'offer', 'soon', 'up']
Размер словаря: 10


## 5. Частоты слов по классам

В сообщениях класса `Spam`:

- `buy`: 2;
- `cheap`: 1;
- `now`: 1;
- `limited`: 1;
- `offer`: 1.

Общее количество слов:

$$
N_{\text{Spam}}=6.
$$

В сообщениях класса `Ham`:

- `meet`: 1;
- `me`: 1;
- `now`: 1;
- `catch`: 1;
- `up`: 1;
- `soon`: 1.

Общее количество слов:

$$
N_{\text{Ham}}=6.
$$

In [ ]:
classes = sorted(data["label"].unique())

word_counts = {}
total_words = {}

for class_name in classes:
    class_texts = data.loc[data["label"] == class_name, "text"]

    counts = Counter()
    for text in class_texts:
        counts.update(tokenize(text))

    word_counts[class_name] = counts
    total_words[class_name] = sum(counts.values())

frequency_table = pd.DataFrame({
    class_name: [
        word_counts[class_name][word]
        for word in vocabulary
    ]
    for class_name in classes
}, index=vocabulary)

frequency_table.index.name = "word"
frequency_table

,ham,spam
word,,
buy,0,2
catch,1,0
cheap,0,1
limited,0,1
me,1,0
meet,1,0
now,1,1
offer,0,1
soon,1,0


In [ ]:
print("Общее количество слов по классам:")
for class_name in classes:
    print(f"{class_name}: {total_words[class_name]}")

Общее количество слов по классам:
ham: 6
spam: 6


## 6. Априорные вероятности классов

В обучающей выборке два спам-сообщения и два обычных сообщения:

$$
P(\text{Spam})=\frac{2}{4}=0.5,
$$

$$
P(\text{Ham})=\frac{2}{4}=0.5.
$$

Это вероятности классов до того, как мы увидели новое сообщение.

In [ ]:
document_counts = Counter(data["label"])
number_of_documents = len(data)

priors = {
    class_name: document_counts[class_name] / number_of_documents
    for class_name in classes
}

priors

{'ham': 0.5, 'spam': 0.5}

## 7. Проблема нулевых вероятностей

Новое сообщение:

$$
d=\text{``buy now''}.
$$

Без сглаживания вероятность слова `buy` в классе `Ham` равна:

$$
P(\text{buy}\mid\text{Ham})=\frac{0}{6}=0.
$$

Тогда:

$$
P(d\mid\text{Ham})
=
P(\text{buy}\mid\text{Ham})
P(\text{now}\mid\text{Ham})
=0.
$$

Одно неизвестное для класса слово полностью обнуляет правдоподобие. Чтобы этого избежать, используется сглаживание Лапласа:

$$
P(w\mid C)=
\frac{N(w,C)+\alpha}
{\sum_{w'\in V}N(w',C)+\alpha|V|}.
$$

При $\alpha=1$:

$$
P(w\mid C)=
\frac{N(w,C)+1}
{N_C+|V|}.
$$

Здесь $N_C=\sum_{w'\in V}N(w',C)$ — общее число слов в документах класса $C$ (в нашем примере 6 для каждого класса).

## 8. Вероятности слов со сглаживанием Лапласа

Для обоих классов:

$$
N_C+\alpha|V|=6+1\cdot10=16.
$$

### Класс Spam

Слово `buy` встретилось два раза:

$$
P(\text{buy}\mid\text{Spam})
=
\frac{2+1}{6+10}
=
\frac{3}{16}.
$$

Слово `now` встретилось один раз:

$$
P(\text{now}\mid\text{Spam})
=
\frac{1+1}{6+10}
=
\frac{2}{16}.
$$

### Класс Ham

Слово `buy` не встретилось ни разу:

$$
P(\text{buy}\mid\text{Ham})
=
\frac{0+1}{6+10}
=
\frac{1}{16}.
$$

Слово `now` встретилось один раз:

$$
P(\text{now}\mid\text{Ham})
=
\frac{1+1}{6+10}
=
\frac{2}{16}.
$$

In [ ]:
alpha = 1
vocabulary_size = len(vocabulary)

def word_probability(word, class_name):
    numerator = word_counts[class_name][word] + alpha
    denominator = (
        total_words[class_name]
        + alpha * vocabulary_size
    )
    return numerator / denominator

for class_name in classes:
    print(f"Класс: {class_name}")
    print(
        "P(buy | class) =",
        word_probability("buy", class_name)
    )
    print(
        "P(now | class) =",
        word_probability("now", class_name)
    )
    print()

Класс: ham
P(buy | class) = 0.0625
P(now | class) = 0.125

Класс: spam
P(buy | class) = 0.1875
P(now | class) = 0.125



## 9. Вычисление правдоподобия

Для сообщения `buy now` каждое слово встретилось один раз. Поэтому:

$$
P(d\mid C)
=
P(\text{buy}\mid C)
P(\text{now}\mid C).
$$

### Spam

$$
P(d\mid\text{Spam})
=
\frac{3}{16}\cdot\frac{2}{16}
=
\frac{6}{256}
=
\frac{3}{128}.
$$

### Ham

$$
P(d\mid\text{Ham})
=
\frac{1}{16}\cdot\frac{2}{16}
=
\frac{2}{256}
=
\frac{1}{128}.
$$

Таким образом, сообщение `buy now` в три раза лучше объясняется классом `Spam`.

## 10. Ненормированные апостериорные оценки

Для классификации сначала вычислим:

$$
\operatorname{score}(C)=P(C)P(d\mid C).
$$

### Spam

$$
\operatorname{score}(\text{Spam})
=
\frac{1}{2}\cdot\frac{3}{128}
=
\frac{3}{256}.
$$

### Ham

$$
\operatorname{score}(\text{Ham})
=
\frac{1}{2}\cdot\frac{1}{128}
=
\frac{1}{256}.
$$

Поскольку:

$$
\frac{3}{256}>\frac{1}{256},
$$

MAP-классификатор выбирает `Spam`.

Однако $\frac{3}{256}$ и $\frac{1}{256}$ — еще не нормированные апостериорные вероятности. Их сумма не равна единице.

## 11. Полная вероятность сообщения

Полная вероятность сообщения равна сумме оценок по всем классам:

$$
P(d)
=
P(d\mid\text{Spam})P(\text{Spam})
+
P(d\mid\text{Ham})P(\text{Ham}).
$$

Подставим значения:

$$
P(d)
=
\frac{3}{256}
+
\frac{1}{256}
=
\frac{4}{256}
=
\frac{1}{64}.
$$

## 12. Апостериорные вероятности

Теперь применим формулу Байеса.

### Вероятность Spam

$$
P(\text{Spam}\mid d)
=
\frac{P(d\mid\text{Spam})P(\text{Spam})}{P(d)}
=
\frac{3/256}{4/256}
=
\frac{3}{4}
=
0.75.
$$

### Вероятность Ham

$$
P(\text{Ham}\mid d)
=
\frac{P(d\mid\text{Ham})P(\text{Ham})}{P(d)}
=
\frac{1/256}{4/256}
=
\frac{1}{4}
=
0.25.
$$

Проверка:

$$
P(\text{Spam}\mid d)+P(\text{Ham}\mid d)
=
0.75+0.25
=
1.
$$

Итог:

$$
\boxed{P(\text{Spam}\mid\text{``buy now''})=0.75}
$$

$$
\boxed{P(\text{Ham}\mid\text{``buy now''})=0.25}
$$

Следовательно:

$$
\boxed{\text{Сообщение классифицируется как Spam}}
$$

In [ ]:
test_message = "buy now"
test_counts = Counter(tokenize(test_message))

likelihoods = {}
scores = {}

for class_name in classes:
    likelihood = 1.0

    for word, count in test_counts.items():
        # Слова, которых нет в обучающем словаре,
        # в данном примере игнорируются.
        if word in vocabulary:
            probability = word_probability(word, class_name)
            likelihood *= probability ** count

    likelihoods[class_name] = likelihood
    scores[class_name] = priors[class_name] * likelihood

evidence = sum(scores.values())

posteriors = {
    class_name: scores[class_name] / evidence
    for class_name in classes
}

result = pd.DataFrame({
    "prior P(C)": priors,
    "likelihood P(d|C)": likelihoods,
    "score P(C)P(d|C)": scores,
    "posterior P(C|d)": posteriors
})

result

,prior P(C),likelihood P(d|C),score P(C)P(d|C),posterior P(C|d)
ham,0.5,0.007812,0.003906,0.25
spam,0.5,0.023438,0.011719,0.75


In [ ]:
predicted_class = max(posteriors, key=posteriors.get)

print("Сообщение:", test_message)
print("Апостериорные вероятности:", posteriors)
print("Предсказанный класс:", predicted_class)

Сообщение: buy now
Апостериорные вероятности: {'ham': 0.25, 'spam': 0.75}
Предсказанный класс: spam


## 13. Зачем нужны логарифмы

Для длинного документа необходимо перемножить большое количество вероятностей:

$$
P(C)P(d\mid C)
=
P(C)\prod_{w\in V}P(w\mid C)^{n_w}.
$$

Такое произведение может стать настолько маленьким, что компьютер округлит его до нуля. Это называется численным переполнением снизу, или underflow.

Поэтому вычисляют логарифм оценки:

$$
\log \operatorname{score}(C)
=
\log P(C)
+
\sum_{w\in V}n_w\log P(w\mid C).
$$

Логарифм является монотонно возрастающей функцией, поэтому:

$$
\arg\max_C \operatorname{score}(C)
=
\arg\max_C \log \operatorname{score}(C).
$$

Для преобразования логарифмических оценок в вероятности используется нормализация:

$$
P(C\mid d)
=
\frac{\exp(s_C)}
{\sum_{C'}\exp(s_{C'})},
$$

где $s_C$ — логарифмическая оценка класса.

In [ ]:
import math

log_scores = {}

for class_name in classes:
    log_score = math.log(priors[class_name])

    for word, count in test_counts.items():
        if word in vocabulary:
            probability = word_probability(word, class_name)
            log_score += count * math.log(probability)

    log_scores[class_name] = log_score

log_scores

{'ham': -5.545177444479562, 'spam': -4.446565155811452}

In [ ]:
# Стабильный вариант softmax:
# перед экспонентой вычитаем максимальное значение.

max_log_score = max(log_scores.values())

exp_scores = {
    class_name: math.exp(log_scores[class_name] - max_log_score)
    for class_name in classes
}

normalizer = sum(exp_scores.values())

log_posteriors = {
    class_name: exp_scores[class_name] / normalizer
    for class_name in classes
}

print("Логарифмические оценки:", log_scores)
print("Апостериорные вероятности:", log_posteriors)

Логарифмические оценки: {'ham': -5.545177444479562, 'spam': -4.446565155811452}
Апостериорные вероятности: {'ham': 0.24999999999999992, 'spam': 0.75}


## 14. Проверка с помощью scikit-learn

В библиотеке `scikit-learn`:

- `CountVectorizer` строит словарь и считает вхождения слов;
- `MultinomialNB` оценивает вероятности классов и слов;
- параметр `alpha=1.0` задает сглаживание Лапласа.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

vectorizer = CountVectorizer()

X_train = vectorizer.fit_transform(data["text"])
y_train = data["label"]

model = MultinomialNB(alpha=1.0)
model.fit(X_train, y_train)

print("Словарь sklearn:")
print(vectorizer.get_feature_names_out())

Словарь sklearn:
['buy' 'catch' 'cheap' 'limited' 'me' 'meet' 'now' 'offer' 'soon' 'up']


In [ ]:
X_test = vectorizer.transform([test_message])

prediction = model.predict(X_test)[0]
probabilities = model.predict_proba(X_test)[0]

probability_by_class = {
    str(class_name): round(float(probability), 4)
    for class_name, probability
    in zip(model.classes_, probabilities)
}

print("Сообщение:", test_message)
print("Классы модели:", model.classes_)
print("Вероятности:", probability_by_class)
print("Предсказанный класс:", prediction)

Сообщение: buy now
Классы модели: ['ham' 'spam']
Вероятности: {'ham': 0.25, 'spam': 0.75}
Предсказанный класс: spam


In [ ]:
# Посмотрим на априорные вероятности, найденные sklearn

sklearn_priors = {
    str(class_name): math.exp(log_prior)
    for class_name, log_prior
    in zip(model.classes_, model.class_log_prior_)
}

sklearn_priors

{'ham': 0.5, 'spam': 0.5}

In [ ]:
# Вероятности слов, найденные sklearn

feature_probabilities = pd.DataFrame(
    data=model.feature_log_prob_,
    index=model.classes_,
    columns=vectorizer.get_feature_names_out()
).map(math.exp)

feature_probabilities

,buy,catch,cheap,limited,me,meet,now,offer,soon,up
ham,0.0625,0.1250,0.0625,0.0625,0.1250,0.1250,0.125,0.0625,0.1250,0.1250
spam,0.1875,0.0625,0.1250,0.1250,0.0625,0.0625,0.125,0.1250,0.0625,0.0625


## 15. Влияние повторений слов

Multinomial Naive Bayes учитывает не только наличие слова, но и количество его повторений.

Например, для сообщения:

> `buy buy buy now`

оценка класса `Spam` содержит:

$$
P(\text{buy}\mid\text{Spam})^3
P(\text{now}\mid\text{Spam}).
$$

Повторение характерного для спама слова `buy` должно сильнее поддержать класс `Spam`.

In [ ]:
messages = [
    "buy now",
    "buy buy buy now",
    "meet now",
    "limited offer",
    "catch up soon"
]

X_messages = vectorizer.transform(messages)
predictions = model.predict(X_messages)
probabilities = model.predict_proba(X_messages)

spam_index = list(model.classes_).index("spam")

prediction_table = pd.DataFrame({
    "message": messages,
    "prediction": predictions,
    "P(spam | message)": probabilities[:, spam_index]
})

prediction_table

,message,prediction,P(spam | message)
0,buy now,spam,0.750000
1,buy buy buy now,spam,0.964286
2,meet now,ham,0.333333
3,limited offer,spam,0.800000
4,catch up soon,ham,0.111111


# Сравнение Multinomial Naive Bayes с другим вариантом Naive Bayes

## 16. Multinomial Naive Bayes и Bernoulli Naive Bayes

Для текстов часто используются два варианта наивного Байеса:

- **Multinomial Naive Bayes** — учитывает количество вхождений слов;
- **Bernoulli Naive Bayes** — учитывает только наличие или отсутствие слов.

### Multinomial Naive Bayes

Признак:

$$
x_w\in\{0,1,2,\ldots\}.
$$

Например:

- `buy now` $\rightarrow x_{\text{buy}}=1$;
- `buy buy buy now` $\rightarrow x_{\text{buy}}=3$.

Правдоподобие:

$$
P(d\mid C)
\propto
\prod_{w\in V}P(w\mid C)^{n_w}.
$$

### Bernoulli Naive Bayes

Признак является бинарным:

$$
x_w\in\{0,1\}.
$$

Повторения слова не учитываются:

- `buy now` $\rightarrow x_{\text{buy}}=1$;
- `buy buy buy now` $\rightarrow x_{\text{buy}}=1$.

Правдоподобие:

$$
P(x\mid C)
=
\prod_{w\in V}
\theta_{C,w}^{x_w}
(1-\theta_{C,w})^{1-x_w},
$$

где:

$$
\theta_{C,w}=P(x_w=1\mid C).
$$

Bernoulli Naive Bayes учитывает не только присутствующие, но и отсутствующие слова.

### Основные различия

| Характеристика | Multinomial NB | Bernoulli NB |
|---|---|---|
| Значение признака | Количество вхождений | Наличие или отсутствие |
| Повторения слов | Учитываются | Не учитываются |
| Типичные данные | BoW, счетчики, иногда TF-IDF | Бинарный BoW |
| Отсутствие слова | Явно обычно не учитывается | Учитывается |
| Подходит для | Более длинных текстов и частот слов | Коротких сообщений и бинарных индикаторов |
| Пример | `free free free` сильнее, чем `free` | `free free free` эквивалентно `free` |

In [ ]:
from sklearn.naive_bayes import BernoulliNB

bernoulli_model = BernoulliNB(alpha=1.0)
bernoulli_model.fit(X_train, y_train)

comparison_messages = [
    "buy now",
    "buy buy buy now",
    "meet now",
    "limited offer",
    "catch up soon"
]

X_comparison = vectorizer.transform(comparison_messages)

mnb_probabilities = model.predict_proba(X_comparison)
bnb_probabilities = bernoulli_model.predict_proba(X_comparison)

mnb_spam_index = list(model.classes_).index("spam")
bnb_spam_index = list(bernoulli_model.classes_).index("spam")

comparison = pd.DataFrame({
    "message": comparison_messages,
    "MultinomialNB P(spam)": mnb_probabilities[:, mnb_spam_index],
    "BernoulliNB P(spam)": bnb_probabilities[:, bnb_spam_index],
    "MultinomialNB prediction": model.predict(X_comparison),
    "BernoulliNB prediction": bernoulli_model.predict(X_comparison)
})

comparison

,message,MultinomialNB P(spam),BernoulliNB P(spam),MultinomialNB prediction,BernoulliNB prediction
0,buy now,0.750000,0.870968,spam,spam
1,buy buy buy now,0.964286,0.870968,spam,spam
2,meet now,0.333333,0.200000,ham,ham
3,limited offer,0.800000,0.870968,spam,spam
4,catch up soon,0.111111,0.027027,ham,ham


### Интерпретация сравнения

Для сообщений:

- `buy now`;
- `buy buy buy now`;

Bernoulli Naive Bayes строит одинаковые бинарные признаки: слово `buy` присутствует, слово `now` присутствует. Поэтому повторение `buy` не дает дополнительной информации.

Multinomial Naive Bayes учитывает, что в первом сообщении `buy` встретилось один раз, а во втором — три раза. Если `buy` характерно для спама, вероятность класса `Spam` может увеличиться.

Таким образом:

- Multinomial NB отвечает на вопрос: **«Сколько раз встретилось слово?»**
- Bernoulli NB отвечает на вопрос: **«Встретилось ли слово хотя бы один раз?»**

## 17. А что насчет Gaussian Naive Bayes?

Gaussian Naive Bayes предполагает, что каждый числовой признак внутри класса имеет нормальное распределение:

$$
P(x_j\mid C)
=
\frac{1}{\sqrt{2\pi\sigma_{C,j}^2}}
\exp\left(
-\frac{(x_j-\mu_{C,j})^2}{2\sigma_{C,j}^2}
\right).
$$

Такой вариант подходит для непрерывных признаков, например:

- возраста;
- температуры;
- давления;
- финансовых показателей;
- результатов медицинских измерений.

Для разреженных счетчиков слов Gaussian Naive Bayes обычно менее естественен, потому что:

1. счетчики слов дискретны, а не непрерывны;
2. большинство элементов BoW-вектора равны нулю;
3. распределение частот слов обычно не является нормальным;
4. для обучения приходится преобразовывать разреженную матрицу в плотную.

Поэтому для обычной текстовой классификации чаще выбирают Multinomial NB или Bernoulli NB.

## 18. Итоги

В этом примере мы вручную классифицировали сообщение `buy now`.

### Априорные вероятности

$$
P(\text{Spam})=0.5,
\qquad
P(\text{Ham})=0.5.
$$

### Правдоподобия

$$
P(d\mid\text{Spam})=\frac{3}{128},
$$

$$
P(d\mid\text{Ham})=\frac{1}{128}.
$$

### Ненормированные оценки

$$
P(\text{Spam})P(d\mid\text{Spam})
=
\frac{3}{256},
$$

$$
P(\text{Ham})P(d\mid\text{Ham})
=
\frac{1}{256}.
$$

### Полная вероятность сообщения

$$
P(d)=\frac{4}{256}.
$$

### Апостериорные вероятности

$$
P(\text{Spam}\mid d)=0.75,
$$

$$
P(\text{Ham}\mid d)=0.25.
$$

Итоговая классификация:

$$
\boxed{\text{``buy now''}\rightarrow\text{Spam}}
$$

### Когда Naive Bayes — хороший выбор

- нужен простой и быстрый базовый алгоритм;
- объем обучающих данных невелик;
- задача — классификация текстов (спам, тематика, тональность);
- признаки представлены как BoW или TF-IDF;
- важны скорость обучения и интерпретируемость.

### Когда лучше выбрать другой метод

- требуется учитывать порядок слов или длинный контекст;
- важны смысловые связи между словами;
- используются эмбеддинги или большие языковые модели;
- нужно максимально возможное качество на сложной задаче.

### Главные выводы

1. Naive Bayes сравнивает, какой класс лучше объясняет наблюдаемый текст.
2. Априорная вероятность отражает распространенность класса до наблюдения текста.
3. Правдоподобие показывает, насколько текст характерен для конкретного класса.
4. Апостериорная вероятность получается после объединения prior и likelihood.
5. Сглаживание Лапласа защищает модель от нулевых вероятностей.
6. На практике вычисления выполняются в логарифмах.
7. Multinomial NB учитывает частоты слов.
8. Bernoulli NB учитывает только присутствие или отсутствие слов.
9. Для разреженных текстовых данных Multinomial NB обычно уместнее Gaussian NB.

В конце ноутбука можно добавить универсальную функцию для ручного предсказания.

In [ ]:
def predict_manual_naive_bayes(message):
    message_counts = Counter(tokenize(message))
    log_scores = {}

    for class_name in classes:
        log_score = math.log(priors[class_name])

        for word, count in message_counts.items():
            if word not in vocabulary:
                continue

            probability = word_probability(word, class_name)
            log_score += count * math.log(probability)

        log_scores[class_name] = log_score

    max_log_score = max(log_scores.values())

    exp_scores = {
        class_name: math.exp(score - max_log_score)
        for class_name, score in log_scores.items()
    }

    normalizer = sum(exp_scores.values())

    probabilities = {
        class_name: score / normalizer
        for class_name, score in exp_scores.items()
    }

    prediction = max(probabilities, key=probabilities.get)

    return {
        "message": message,
        "prediction": prediction,
        "probabilities": probabilities,
        "log_scores": log_scores
    }


examples = [
    "buy now",
    "cheap offer",
    "meet me now",
    "buy buy limited offer",
    "catch up"
]

for message in examples:
    result = predict_manual_naive_bayes(message)
    print(result)

{'message': 'buy now', 'prediction': 'spam', 'probabilities': {'ham': 0.24999999999999992, 'spam': 0.75}, 'log_scores': {'ham': -5.545177444479562, 'spam': -4.446565155811452}}
{'message': 'cheap offer', 'prediction': 'spam', 'probabilities': {'ham': 0.19999999999999996, 'spam': 0.8}, 'log_scores': {'ham': -6.238324625039508, 'spam': -4.852030263919617}}
{'message': 'meet me now', 'prediction': 'ham', 'probabilities': {'ham': 0.8, 'spam': 0.19999999999999996}, 'log_scores': {'ham': -6.931471805599452, 'spam': -8.317766166719343}}
{'message': 'buy buy limited offer', 'prediction': 'spam', 'probabilities': {'ham': 0.02702702702702705, 'spam': 0.9729729729729728}, 'log_scores': {'ham': -11.78350206951907, 'spam': -8.199983131062961}}
{'message': 'catch up', 'prediction': 'ham', 'probabilities': {'ham': 0.8, 'spam': 0.19999999999999996}, 'log_scores': {'ham': -4.852030263919617, 'spam': -6.238324625039508}}


> В этой учебной реализации слова, отсутствующие в обучающем словаре, игнорируются — так же ведет себя `CountVectorizer` при преобразовании новых сообщений. Для реального проекта также нужны нормальная токенизация, обработка регистра, знаков препинания, URL, чисел и других особенностей текста.